In [ ]:
!pip install -q groq

import json
import re
import time
from groq import Groq
from google.colab import drive

# --- 1. Configure Groq ---
GROQ_API_KEY = ""
client = Groq(api_key=GROQ_API_KEY)

MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

# --- 2. Load Data ---
input_file = 'g24_full_multilingual_data.json'
output_file = 'g24_final_results_groq.json'

try:
    with open(output_file, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    print(" The current progress has been detected. Continue processing...")
except FileNotFoundError:
    with open(input_file, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    print("Start a new generation task...")

# --- 3. helper function ---
def extract_number(text):
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", str(text))
    return nums[-1] if nums else text

def generate_with_groq(prompt):
    """A Groq call function with current-limiting protection"""

    try:
        completion = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=512,
            stream=False

        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        if "429" in str(e):
            print(" A Groq call function with current-limiting protection..")
            time.sleep(10)
            return generate_with_groq(prompt)
        print(f"error: {e}")
        return f"ERROR: {str(e)}"

# --- 4. Major Loop ---
print(f" Start the Groq acceleration mode and target the model: {MODEL_ID}")

for i, entry in enumerate(dataset):
    is_done = True
    for lang in ['en', 'zh', 'es']:
        out = entry['languages'][lang].get('model_output', [])
        if len(out) < 3:
            is_done = False
            break
    if is_done: continue




        existing_outputs = [o for o in entry['languages'][lang].get('model_output', []) if "ERROR" not in str(o)]
        needed = 3 - len(existing_outputs)

        if needed > 0:
            new_outputs = []
            for _ in range(needed):
                res = generate_with_groq(prompt)
                new_outputs.append(res)
                time.sleep(1)

            entry['languages'][lang]['model_output'] = existing_outputs + new_outputs

        if entry['type'] == 'MGSM' and entry['languages'][lang]['model_output']:
            gt = str(entry['ground_truth']).strip()
            pred = extract_number(entry['languages'][lang]['model_output'][0])
            entry['languages'][lang]['is_correct'] = (pred == gt)

    # real time saving
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    if i % 10 == 0:
        print(f" : {i+1}/{len(dataset)} (ID: {entry['id']} filed)")

print(f" Data storage : {output_file}")

In [ ]:
import json
import re
import time
from groq import Groq


GROQ_API_KEY = ""
client = Groq(api_key=GROQ_API_KEY)
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

input_file = 'g24_full_multilingual_data.json'
output_file = 'g24_final_results_groq - 副本.json'


try:
    with open(output_file, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    print("The current progress has been detected. Continue processing...")
except FileNotFoundError:
    with open(input_file, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    print("Start a brand new generation task...")


def extract_number(text):
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", str(text))
    return nums[-1] if nums else text

def generate_with_retry(prompt, max_retries=3):

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=512
            )
            time.sleep(3)
            return completion.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                print(f"In rate limiting. Take a 30-second break... "Detailed: {e})")
                time.sleep(35)
            elif "503" in str(e):
                time.sleep(10)
            else:
                print(f"Error: {e}")
                time.sleep(5)
    return "ERROR"



In [ ]:
print(f"Start the generation engine. Target:{len(dataset)}...")


for i, entry in enumerate(dataset):

    is_id_complete = True
    for lang in ['en', 'zh', 'es']:
        out = entry['languages'][lang].get('model_output', [])
        valid_out = [o for o in out if "ERROR" not in str(o)]
        if len(valid_out) < 3:
            is_id_complete = False
            break

    if is_id_complete: continue


    for lang in ['en', 'zh', 'es']:
        context = entry.get('context_en', "")
        question = entry['languages'][lang]['question']


        if entry['type'] == 'XQuAD' and context:

            if lang == 'en':
                prompt = f"Context: {context}\nQuestion: {question}\nAnswer concisely."
            elif lang == 'zh':
                prompt = f"背景信息: {context}\n问题: {question}\n请根据背景简洁回答。"
            else:
                prompt = f"Contexto: {context}\nPregunta: {question}\nResponde concisamente."
        else:

            prompt = entry['languages'][lang]['prompt']


        current_out = entry['languages'][lang].get('model_output', [])
        valid_out = [o for o in current_out if "ERROR" not in str(o)]

        needed = 3 - len(valid_out)
        if needed > 0:
            print(f"processing {entry['id']} [{lang}]，still needs to be supplemented {needed} ...")
            new_results = []
            for _ in range(needed):
                res = generate_with_retry(prompt)
                if "ERROR" not in res:
                    new_results.append(res)


            entry['languages'][lang]['model_output'] = valid_out + new_results


        if entry['type'] == 'MGSM' and entry['languages'][lang]['model_output']:
            gt = str(entry['ground_truth']).strip()
            pred = extract_number(entry['languages'][lang]['model_output'][0])
            entry['languages'][lang]['is_correct'] = (pred == gt)


    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    if i % 5 == 0:
        print(f"real-time progress: {i+1}/{len(dataset)} (Newly completed {entry['id']})")

print(f"The final outcome has been saved to: {output_file}")